In [3]:
!pip install -q google-genai scikit-learn numpy

In [4]:

from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets")

client = genai.Client(api_key=api_key)

MODEL = "gemini-3.7-flash"

print("✅ Gemini API connected successfully!")

✅ Gemini API connected successfully!


In [5]:

import json
import ast
import operator
import re

from datetime import datetime

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:

documents = [
    "Artificial intelligence enables computers to perform tasks that normally require human intelligence.",

    "Machine learning allows computers to learn patterns from data without being explicitly programmed.",

    "Deep learning uses neural networks with multiple layers to learn complex patterns from large datasets.",

    "Natural language processing enables computers to understand and process human language.",

    "Computer vision allows machines to understand and analyze images and videos.",

    "Supervised learning trains a model using labeled training data.",

    "Unsupervised learning finds hidden patterns in data without labeled examples.",

    "Reinforcement learning trains an agent through rewards and penalties based on its actions.",

    "A neural network consists of interconnected artificial neurons organized into layers.",

    "Convolutional neural networks are commonly used for image classification and visual recognition.",

    "Recurrent neural networks are designed to process sequential and time dependent data.",

    "Transformers use attention mechanisms to process relationships between tokens efficiently.",

    "Large language models are trained on massive text datasets to understand and generate natural language.",

    "Generative AI models can create text, images, audio, video, and other types of content.",

    "Overfitting occurs when a machine learning model learns the training data too closely and performs poorly on new data.",

    "Underfitting occurs when a model is too simple to capture important patterns in the training data.",

    "Precision measures the proportion of predicted positive examples that are actually positive.",

    "Recall measures the proportion of actual positive examples that a model successfully identifies.",

    "F1 score combines precision and recall into a single evaluation metric.",

    "A confusion matrix shows the number of correct and incorrect predictions for different classes.",

    "Feature engineering creates or transforms input variables to improve machine learning performance.",

    "Data preprocessing includes cleaning, transforming, and preparing data before model training.",

    "Cross validation evaluates a machine learning model using multiple training and validation splits.",

    "Gradient descent is an optimization algorithm used to minimize a model's loss function.",

    "An embedding represents words, sentences, or documents as numerical vectors that capture semantic relationships."
]

print("Number of documents:", len(documents))

Number of documents: 25


In [7]:

class Pipeline:

    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english"
        )

    def preprocess(self, text):

        text = text.lower()

        text = re.sub(
            r"[^a-zA-Z0-9\s]",
            "",
            text
        )

        text = re.sub(
            r"\s+",
            " ",
            text
        ).strip()

        return text

    def fit_transform(self, documents):

        cleaned_docs = [
            self.preprocess(doc)
            for doc in documents
        ]

        return self.vectorizer.fit_transform(
            cleaned_docs
        )

    def transform(self, text):

        cleaned_text = self.preprocess(text)

        return self.vectorizer.transform(
            [cleaned_text]
        )


pipeline = Pipeline()

corpus_matrix = pipeline.fit_transform(documents)

print("✅ TF-IDF pipeline ready")
print("Matrix shape:", corpus_matrix.shape)

✅ TF-IDF pipeline ready
Matrix shape: (25, 157)


In [8]:

def search_docs(query, top_k=3):

    query_vector = pipeline.transform(query)

    similarities = cosine_similarity(
        query_vector,
        corpus_matrix
    )[0]

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    results = []

    for index in top_indices:

        score = float(similarities[index])

        if score > 0:

            results.append({
                "document_id": int(index + 1),
                "document": documents[index],
                "similarity_score": round(score, 4)
            })

    if not results:

        return {
            "found": False,
            "message": "No relevant document found."
        }

    return {
        "found": True,
        "results": results
    }

In [ ]:

print(
    json.dumps(
        search_docs("What is deep learning?"),
        indent=2
    )
)

In [9]:

def calculator(expression):

    try:

        allowed_operators = {
            ast.Add: operator.add,
            ast.Sub: operator.sub,
            ast.Mult: operator.mul,
            ast.Div: operator.truediv,
            ast.Pow: operator.pow,
            ast.Mod: operator.mod,
            ast.USub: operator.neg,
            ast.UAdd: operator.pos
        }

        def evaluate(node):

            if isinstance(node, ast.Constant):

                if isinstance(
                    node.value,
                    (int, float)
                ):
                    return node.value

                raise ValueError("Invalid value")

            if isinstance(node, ast.BinOp):

                left = evaluate(node.left)
                right = evaluate(node.right)

                operation = allowed_operators.get(
                    type(node.op)
                )

                if operation is None:
                    raise ValueError(
                        "Operator not allowed"
                    )

                return operation(left, right)

            if isinstance(node, ast.UnaryOp):

                value = evaluate(node.operand)

                operation = allowed_operators.get(
                    type(node.op)
                )

                if operation is None:
                    raise ValueError(
                        "Operator not allowed"
                    )

                return operation(value)

            raise ValueError(
                "Invalid expression"
            )

        tree = ast.parse(
            expression,
            mode="eval"
        )

        result = evaluate(tree.body)

        return {
            "success": True,
            "result": result
        }

    except Exception as e:

        return {
            "success": False,
            "error": str(e)
        }

In [ ]:
print(calculator("80 / 100 * 100"))

In [ ]:

def dispatch_action(
    action_name,
    parameters
):

    if action_name not in tool_registry:

        return {
            "success": False,
            "error": f"Unknown tool: {action_name}"
        }

    try:

        if action_name == "search_docs":

            query = parameters.get("query")

            if not query:

                return {
                    "success": False,
                    "error": "query is required"
                }

            return search_docs(query)

        elif action_name == "calculator":

            expression = parameters.get(
                "expression"
            )

            if not expression:

                return {
                    "success": False,
                    "error": "expression is required"
                }

            return calculator(expression)

        elif action_name == "get_today":

            return get_today()

    except Exception as e:

        return {
            "success": False,
            "error": str(e)
        }

In [10]:

SYSTEM_PROMPT = """
You are a ReAct AI Agent.

Solve the user's problem by using the available tools.

TOOLS:

1. search_docs

Purpose:
Search the Day 11 AI/ML knowledge base.

Parameters:
{
    "query": "machine learning"
}


2. calculator

Purpose:
Perform arithmetic calculations.

Parameters:
{
    "expression": "80 / 100 * 100"
}


3. get_today

Purpose:
Return today's date.

Parameters:
{}


RULES:

- Use search_docs for knowledge-base questions.
- Use calculator for arithmetic.
- Use get_today for today's date.
- You may use multiple tools in sequence.
- Never invent a tool result.
- Always use the actual Observation.
- Do not pretend a tool was executed.
- Stop when the problem is solved.
- Maximum 8 steps.

Do not expose private chain-of-thought.

The "thought" field should only be a short action summary.

Return ONLY valid JSON.

ACTION FORMAT:

{
    "type": "action",
    "thought": "Brief reason for selecting the tool",
    "action": "search_docs",
    "parameters": {
        "query": "..."
    }
}

FINAL FORMAT:

{
    "type": "final",
    "answer": "Final answer"
}
"""

In [11]:

def ask_gemini(conversation):

    prompt = SYSTEM_PROMPT

    prompt += "\n\nCONVERSATION:\n"

    for message in conversation:

        prompt += (
            f"\n{message['role'].upper()}:\n"
        )

        prompt += message["content"]

        prompt += "\n"

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text

In [ ]:

def clean_json(text):

    text = text.strip()

    if text.startswith("```json"):

        text = text[7:]

        if text.endswith("```"):
            text = text[:-3]

    elif text.startswith("```"):

        text = text[3:]

        if text.endswith("```"):
            text = text[:-3]

    return text.strip()

In [12]:

def run_agent(question, max_steps=8):

    print("\n")
    print("=" * 70)
    print("QUESTION")
    print("=" * 70)
    print(question)

    conversation = [
        {
            "role": "user",
            "content": question
        }
    ]

    previous_actions = []

    for step in range(1, max_steps + 1):

        print("\n")
        print("-" * 70)
        print(f"STEP {step}")
        print("-" * 70)

        try:

            response = ask_gemini(
                conversation
            )

            print("\nGEMINI:")
            print(response)

            clean_response = clean_json(
                response
            )

            data = json.loads(
                clean_response
            )

        except json.JSONDecodeError:

            print("\n❌ Invalid JSON")

            conversation.append({
                "role": "user",
                "content": """
Your previous response was not valid JSON.

Return ONLY valid JSON.
"""
            })

            continue

        except Exception as e:

            print(
                f"\n❌ Gemini API Error: {e}"
            )

            return None

        # ----------------------------------------------------
        # FINAL ANSWER
        # ----------------------------------------------------

        if data.get("type") == "final":

            print("\n")
            print("=" * 70)
            print("FINAL ANSWER")
            print("=" * 70)

            answer = data.get(
                "answer",
                "No answer generated."
            )

            print(answer)

            return answer

        # ----------------------------------------------------
        # ACTION
        # ----------------------------------------------------

        if data.get("type") != "action":

            print(
                "\n❌ Invalid response type"
            )

            continue

        thought = data.get(
            "thought",
            ""
        )

        action = data.get(
            "action"
        )

        parameters = data.get(
            "parameters",
            {}
        )

        print("\nTHOUGHT:")
        print(thought)

        print("\nACTION:")
        print(action)

        print("\nPARAMETERS:")
        print(
            json.dumps(
                parameters,
                indent=2
            )
        )

        # ----------------------------------------------------
        # LOOP DETECTION
        # ----------------------------------------------------

        signature = (
            action,
            json.dumps(
                parameters,
                sort_keys=True
            )
        )

        if signature in previous_actions:

            print(
                "\n⚠️ Repeated action detected!"
            )

            conversation.append({
                "role": "user",
                "content": """
You already performed this exact action.

Do not repeat it.
Use the previous observation and continue.
"""
            })

            continue

        previous_actions.append(
            signature
        )

        # ----------------------------------------------------
        # EXECUTE TOOL
        # ----------------------------------------------------

        observation = dispatch_action(
            action,
            parameters
        )

        print("\nOBSERVATION:")

        print(
            json.dumps(
                observation,
                indent=2
            )
        )

        # ----------------------------------------------------
        # SEND OBSERVATION BACK
        # ----------------------------------------------------

        conversation.append({
            "role": "assistant",
            "content": response
        })

        conversation.append({
            "role": "user",
            "content": (
                "OBSERVATION FROM TOOL:\n"
                + json.dumps(observation)
                + "\n\n"
                "This is the real tool output. "
                "Do not invent or modify it. "
                "Continue solving the original question."
            )
        })

    print("\n")
    print("=" * 70)
    print("AGENT STOPPED")
    print("=" * 70)

    print(
        "Maximum number of steps reached."
    )

    return None

In [13]:

run_agent(
    """
Search the Day 11 knowledge base and find
what an embedding is.

Then calculate how many numerical values
are present if there are 10 embedding vectors
and each vector has 100 dimensions.
"""
)



QUESTION

Search the Day 11 knowledge base and find
what an embedding is.

Then calculate how many numerical values
are present if there are 10 embedding vectors
and each vector has 100 dimensions.



----------------------------------------------------------------------
STEP 1
----------------------------------------------------------------------

❌ Gemini API Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [14]:

test_questions = [

    """
Search the Day 11 knowledge base and find
what an embedding is.

Then calculate how many numerical values
are present if there are 10 embedding vectors
and each vector has 100 dimensions.
""",

    """
Search the Day 11 knowledge base and find
the definition of precision.

Then calculate the precision if 80 predicted
positive examples are actually positive out
of 100 predicted positive examples.
""",

    """
Search the Day 11 knowledge base and find
the definition of recall.

Then calculate the recall if a model correctly
identifies 75 actual positive examples out
of 100 actual positive examples.
""",

    """
First get today's date.

Then search the Day 11 knowledge base and find
what gradient descent is used for.

Finally calculate 25 * 4.
""",

    """
First search the Day 11 knowledge base and find
what deep learning is.

Then search the knowledge base and find what
transformers use to process relationships
between tokens.

Finally calculate the total number of documents
if 2 searches each return 3 documents.
"""
]